In [1]:
# Package installs go here 
%conda install polars, plotly

/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.html

  deprecated.topic(
/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.

In [2]:
# python imports 
import polars as pl
import plotly.io as pio
# VS Code draws the plotly mimetype natively (fast); the HTML copy keeps charts in the slides export
pio.renderers.default = "plotly_mimetype+notebook_connected"

In [3]:
# Read eBird csv data
ebird = pl.read_csv("data/ebird.csv") 
observations = pl.read_csv("data/observation.csv") 
species = pl.read_csv("data/899_Species.csv") 
header = pl.read_csv(
    "data/header.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)
locations = pl.read_csv(
    "data/location.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)

traits = pl.read_csv(
    "data/traits.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
)
synonyms = pl.read_csv("data/synonyms.csv")


In [4]:
%%capture
# Rename variable names in eBird 
# In polars, we can apply a dictionary to rename columns
ebird = ebird.rename({"K-Latitude" : "key_latitude", "K-Longitude": "key_longitude"})
ebird = ebird.rename({"Latitude" : "latitude", "Longitude": "longitude"})

In [5]:
# Helper function that converts form degree:minutes to degree:decimal
def ddmm_to_dd(column_name: str) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) - degree * 100
    return degree + (minutes / 60)

ebird = ebird.with_columns(
    key_latitude = -ddmm_to_dd("key_latitude"), 
    key_longitude = ddmm_to_dd("key_longitude")
)

ebird = ebird.with_columns(
    latitude = -ddmm_to_dd("latitude"), 
    longitude = ddmm_to_dd("longitude")
)




In [6]:
# Change ebird start and end dates from string to date type (format is YYYY/MM/DD)
ebird = ebird.with_columns(
    pl.col("Observation Start Date", "Observation End Date").str.to_date("%Y/%m/%d")
)

## Slide 1 - Sydney Heatmap
The following graph is a heatmap of bird occurences in sydney...

In [7]:
import plotly.figure_factory as ff

# Approximate Greater Sydney extent (Blue Mountains to coast, Central Coast to Wollondilly)
SYD_LAT = (-34.35, -32.99)
SYD_LON = (149.97, 151.63)

sydney_ebird = ebird.filter(
    pl.col("key_latitude").is_between(*SYD_LAT),
    pl.col("key_longitude").is_between(*SYD_LON),
)

fig = ff.create_hexbin_mapbox(
    data_frame=sydney_ebird, lat="key_latitude", lon="key_longitude",
    nx_hexagon=10, opacity=0.5, labels={"color": "Point Count"},
    mapbox_style="open-street-map"
)

fig.update_layout(margin = dict(b = 0, t = 0, l = 0, r = 0)) 
fig.show()


The hotspots are wollemi national park, hornsby and wollogong.

## eBird Observation Histogram

In [11]:
# Code for observation histogram
import plotly.express as px

# Count eBird observations per year, using the observation start date
obs_per_year = (
    ebird
    .group_by(pl.col("Observation Start Date").dt.year().alias("Year"))
    .len(name="Observations")
    .sort("Year")
)

In [12]:
# Frequency table of eBird observations per year, in a scrollable box
from IPython.display import HTML

freq_table = (
    ebird
    .group_by(pl.col("Observation Start Date").dt.year().alias("Year"))
    .len(name="Observations")
    .sort("Year")
    .with_columns(
        (pl.col("Observations") / pl.col("Observations").sum() * 100).round(2).alias("Percent")
    )
)

# Show every row, then wrap the table in a box that scrolls once it is 400px tall
with pl.Config(tbl_rows=-1):
    table_html = freq_table._repr_html_()

HTML(f'<div style="max-height: 400px; overflow-y: auto;">{table_html}</div>')

Year,Observations,Percent
i32,u32,f64
1951,5,0.0
1952,293,0.02
1953,265,0.01
1954,269,0.01
1955,212,0.01
1956,221,0.01
1957,186,0.01
1958,64,0.0
1962,13,0.0


- The *earliest* observation: 27th August 1951 
- The *latest* observation start is 31st March 2020 

- Dates are strings

## eBird heatmap

In [15]:
# Code for eBird Heatmap
import plotly.express as px

# NSW extent (same bounds as the R notebook)
NSW_LAT = (-37.7, -28.0)
NSW_LON = (140.5, 164.0)

# eBird key coordinates are the centres of 10-minute Bird Atlas grid cells,
# so each tile reaches 5 minutes either side of its centre
HALF_CELL = 5 / 60

# Count observations in each grid cell
nsw_cells = (
    ebird
    .filter(
        pl.col("key_latitude").is_between(*NSW_LAT),
        pl.col("key_longitude").is_between(*NSW_LON),
    )
    .group_by("key_latitude", "key_longitude")
    .len(name="Observations")
    .with_row_index("cell_id")
    .with_columns(log_observations=pl.col("Observations").log10())
)

# Turn each grid cell into a square polygon so plotly can draw it as a tile
def cell_square(lat, lon):
    return [[
        [lon - HALF_CELL, lat - HALF_CELL],
        [lon + HALF_CELL, lat - HALF_CELL],
        [lon + HALF_CELL, lat + HALF_CELL],
        [lon - HALF_CELL, lat + HALF_CELL],
        [lon - HALF_CELL, lat - HALF_CELL],
    ]]

grid_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": row["cell_id"],
            "geometry": {"type": "Polygon", "coordinates": cell_square(row["key_latitude"], row["key_longitude"])},
        }
        for row in nsw_cells.iter_rows(named=True)
    ],
}

# Plasma runs yellow -> orange -> pink -> purple -> deep blue, changing hue as well as
# lightness so neighbouring values are easy to tell apart. It is also colourblind-safe.
# Reversed so yellow = few observations and deep blue = many.
COLOUR_SCALE = px.colors.sequential.Plasma_r

# Counts range from 1 to over 100,000, so colour by log10 and label the scale in real counts
max_log = nsw_cells["log_observations"].max()
tick_values = list(range(0, int(max_log) + 1))
tick_labels = [f"{10 ** v:,}" for v in tick_values]

fig = px.choropleth_mapbox(
    nsw_cells,
    geojson=grid_geojson,
    locations="cell_id",
    color="log_observations",
    color_continuous_scale=COLOUR_SCALE,
    range_color=(0, max_log),
    custom_data=["Observations", "key_latitude", "key_longitude"],
    mapbox_style="open-street-map",
    center={"lat": -32.8, "lon": 147.0},
    zoom=4.6,
    opacity=0.85,
    title="Heatmap of eBird observations in NSW",
)
fig.update_traces(
    marker_line_width=0,
    hovertemplate="%{customdata[0]:,} observations<br>Cell centre: %{customdata[1]:.2f}, %{customdata[2]:.2f}<extra></extra>",
)
fig.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title="Observations", tickvals=tick_values, ticktext=tick_labels),
)
fig.show()

## DES Notes 
- Number of observations in the DES Data  
- As this data is more recent, compare reporting rates to pre-2020 surveys 
- Are there any previously reported species missing in the DES data. 
- Where are the DES reports concentrated. 
- Isolate and analyze interesting / popular species 
- The des data has region codes
- It's an xslx file but the first table has everything we need to know about each survey. 
- 


In [16]:
# Reading DES data: only the first sheet of the spreadsheet
# engine="openpyxl" because polars' default Excel reader (fastexcel) isn't installed
des = pl.read_excel(
    "data/des.xlsx",
    sheet_name="DES_Backlog_Format",
    engine="openpyxl",
)
des

Survey ID,DES Ref,Latitude,Longitude,Observation Start Date,Observation End Date,Locality,Comments,Collection flag,Observer Id,Observer Last Name,Observer First Name,Location name,Region code,Species Common Name,Species ID,Breeding Code,Quality flag,Species Comments [Notes],K-Latitude,K-Longitude
i64,str,i64,i64,datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,null,str,str,str
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Emu""","""1""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Masked Lapwing""","""133""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australian Wood Duck""","""202""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Pacific Black Duck""","""208""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australasian Grebe""","""61""",null,null,"""NULL""","""2945""","""15035"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Stubble Quail""","""9""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Australian Magpie""","""705""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Willie Wagtail""","""364""",null,null,"""NULL""","""3155""","""14925"""


In [17]:
# Converts DES coordinates from degrees-minutes (DDMM) to decimal degrees
# The K- columns are a cleaned copy of Latitude/Longitude: they fix small typos
# (e.g. 15034 -> 15035) and hold the text "NULL" where the original was unusable.
des = des.rename({
    "K-Latitude": "key_latitude", "K-Longitude": "key_longitude",
    "Latitude": "latitude", "Longitude": "longitude",
})

# Turn the text "NULL" into a real missing value, then make the key columns numbers
des = des.with_columns(
    pl.col("key_latitude", "key_longitude").replace("NULL", None).cast(pl.Int64)
)

# A DDMM value is only a real coordinate if its minutes are below 60 and its degrees
# are inside Australia's range. Anything else (e.g. 30015 or 15275) becomes missing.
def valid_ddmm(column_name: str, min_degree: int, max_degree: int) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) % 100
    return (minutes < 60) & degree.is_between(min_degree, max_degree)

des = des.with_columns(
    pl.when(valid_ddmm(c, 9, 45)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["latitude", "key_latitude"]
).with_columns(
    pl.when(valid_ddmm(c, 112, 160)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["longitude", "key_longitude"]
)

# Same conversion as for eBird: latitude is south, so it becomes negative
des = des.with_columns(
    key_latitude=-ddmm_to_dd("key_latitude"),
    key_longitude=ddmm_to_dd("key_longitude"),
    latitude=-ddmm_to_dd("latitude"),
    longitude=ddmm_to_dd("longitude"),
)

des.select("latitude", "longitude", "key_latitude", "key_longitude").describe()

statistic,latitude,longitude,key_latitude,key_longitude
str,f64,f64,f64,f64
"""count""",104103.0,104125.0,104103.0,104125.0
"""null_count""",37.0,15.0,37.0,15.0
"""mean""",-31.436353,151.045275,-31.436357,151.045286
"""std""",1.804477,2.290407,1.804481,2.290405
"""min""",-37.416667,141.083333,-37.416667,141.083333
"""25%""",-33.25,149.916667,-33.25,149.916667
"""50%""",-31.083333,151.416667,-31.083333,151.416667
"""75%""",-30.083333,152.916667,-30.083333,152.916667
"""max""",-21.083333,153.583333,-21.083333,153.583333
